# SDSS Spectra: Preprocessing from Local FITS Files
### Companion to `reproduce_figure3_ppplot.ipynb` — Jespersen et al. (2025)

---

## What changed from the previous version

The previous version downloaded spectra on-the-fly via `astroquery`.
This version adds a **`local_fits` mode** that reads pre-downloaded FITS files,
which is much faster and resumable for the full 510k dataset.

**Run `data_download.ipynb` first** to populate `sdss_raw/fits/`.

## Data source modes

| `DATA_SOURCE` | Reads from | Best for |
|---|---|---|
| `"local_fits"` | `sdss_raw/fits/**/*.fits` | Full scale (after download) ← **recommended** |
| `"astroquery"` | SDSS SkyServer live | Quick tests (≤ 1000 spectra) |

## Outputs (unchanged)

| File | Shape | Content |
|---|---|---|
| `spectra_obs_frame.npy` | (N, 3921) | Observed-frame, for spender |
| `spectra_restframe_norm.npy` | (N, 3600) | Rest-frame normalised, for PCA |
| `spectra_restframe_unnorm.npy` | (N, 3600) | Rest-frame unnormalised |
| `norm_consts.npy` | (N,) | Median flux 5300–5850 Å |
| `spectra_matrix.npy` | (N, 3600) | Alias of restframe_norm (backwards compat) |
| `metadata.npy` | (N,) | specobjid, z, petroMag_r … |
| `wave_obs_grid.npy` | (3921,) | spender observed-frame wavelength axis |
| `wave_grid.npy` | (3600,) | Rest-frame wavelength axis |


## Cell 0 — Imports

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.interpolate import interp1d
import time, sys, logging

import astropy.io.fits as fits   # for reading local FITS files directly

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s",
                    datefmt="%H:%M:%S")
log = logging.getLogger("SDSSPrep")

%matplotlib inline
plt.rcParams.update({"figure.dpi": 120, "font.size": 11})
print(f"Python {sys.version.split()[0]}  |  NumPy {np.__version__}")
print("All imports OK ✓")


## Cell 1 — Configuration

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# SCALE KNOBS — change DATA_SOURCE to switch modes
# ═══════════════════════════════════════════════════════════════════════
DATA_SOURCE = "local_fits"    # "local_fits"  ← use pre-downloaded FITS files (recommended)
                               # "astroquery"  ← live download (small N only)
N_SPECTRA   = None             # None = all available. Set to e.g. 500 for quick tests.

# ── Paths ──────────────────────────────────────────────────────────────
# For local_fits: where data_download.ipynb saved the FITS files
FITS_DIR     = Path("sdss_raw/fits")
META_CSV     = Path("sdss_raw/sdss_metadata.csv")   # produced by data_download.ipynb

# For astroquery: use the SDSS HTTP API directly
Z_MAX        = 0.5

# ── Output directory (shared with other notebooks) ─────────────────────
OUT_DIR = Path("sdss_output"); OUT_DIR.mkdir(exist_ok=True)
# ═══════════════════════════════════════════════════════════════════════

# ── Wavelength grids (do not change without also updating spender) ─────

# spender's SDSS observed-frame grid: 10^arange(3.578, 3.97, 0.0001)
# 3921 log-spaced pixels from 3784–9333 Å.
WAVE_OBS_LOGLAM = np.arange(3.578, 3.970 + 0.00005, 0.0001)
wave_obs_grid   = 10 ** WAVE_OBS_LOGLAM         # shape (3921,)
N_PIXELS_OBS    = len(wave_obs_grid)

# Rest-frame linear grid for normalised spectra / PCA
WAVE_REST_MIN = 3800.0
WAVE_REST_MAX = 9200.0
N_PIXELS_REST = 3600
wave_rest_grid = np.linspace(WAVE_REST_MIN, WAVE_REST_MAX, N_PIXELS_REST)

# normalisation window (paper §3.1)
NORM_WAVE_MIN = 5300.0
NORM_WAVE_MAX = 5850.0

# SDSS sky emission lines in observer frame (Å) — used for the rest-frame grids
SKYLINES_OBS    = np.array([
    5577.3, 5889.9, 6157.0, 6300.3, 6363.8,
    6834.3, 6864.0, 7340.9, 7523.7, 7715.0,
    8344.6, 8399.0, 8430.0, 8791.2, 8827.1, 9375.0,
])
SKYLINE_WIDTH_AA = 10.0

log.info(f"DATA_SOURCE='{DATA_SOURCE}'  N_SPECTRA={N_SPECTRA}")
log.info(f"obs-frame grid: {N_PIXELS_OBS} pix ({wave_obs_grid[0]:.0f}–{wave_obs_grid[-1]:.0f} Å)")
log.info(f"rest-frame grid: {N_PIXELS_REST} pix ({WAVE_REST_MIN:.0f}–{WAVE_REST_MAX:.0f} Å)")


## Cell 2 — Load Galaxy Metadata

In [ ]:
def load_metadata_from_csv(csv_path, n=None):
    """
    Load the galaxy metadata CSV produced by data_download.ipynb.

    Parameters
    ----------
    csv_path : Path
        Path to sdss_raw/sdss_metadata.csv
    n : int or None
        Limit to first n rows.
    """
    if not csv_path.exists():
        raise FileNotFoundError(
            f"{csv_path} not found.\n"
            "Run data_download.ipynb first to download metadata."
        )
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.lower().str.strip()
    if n is not None:
        df = df.head(n)
    log.info(f"Loaded metadata: {len(df):,} rows from {csv_path}")
    return df


def load_metadata_from_astroquery(n, z_max):
    """
    Fallback: query SDSS SkyServer live via astroquery.
    Good for small N (≤ 1000) when you do not have the metadata CSV yet.
    """
    import requests, io
    sql = f"""SELECT TOP {n}
    s.specobjid, s.plate, s.mjd, s.fiberid,
    s.z AS redshift, s.zwarning, s.subclass AS specsubclass, p.petroMag_r
FROM SpecObj AS s JOIN PhotoObj AS p ON s.bestobjid = p.objid
WHERE s.class = 'GALAXY' AND s.z BETWEEN 0.01 AND {z_max}
  AND s.zwarning = 0 AND p.petroMag_r BETWEEN 14.0 AND 17.8
  AND p.petroMag_r != -9999
ORDER BY s.specobjid"""

    for dr in [18, 17, 16]:
        url = f"https://skyserver.sdss.org/dr{dr}/SkyServerWS/SearchTools/SqlSearch"
        try:
            resp = requests.get(url, params={"cmd": sql, "format": "csv"}, timeout=120)
            text = resp.text.strip()
            if "error" not in text[:100].lower() and "<html" not in text[:100].lower():
                df = pd.read_csv(io.StringIO(text))
                if len(df) > 0:
                    df.columns = df.columns.str.lower().str.strip()
                    log.info(f"astroquery DR{dr}: {len(df):,} rows")
                    return df
        except Exception:
            continue
    raise RuntimeError("astroquery failed. Run data_download.ipynb instead.")


if DATA_SOURCE == "local_fits":
    df_meta = load_metadata_from_csv(META_CSV, n=N_SPECTRA)
elif DATA_SOURCE == "astroquery":
    if N_SPECTRA is None or N_SPECTRA > 2000:
        raise ValueError("Set N_SPECTRA ≤ 2000 when using astroquery. "
                         "Use DATA_SOURCE='local_fits' for larger runs.")
    df_meta = load_metadata_from_astroquery(N_SPECTRA, Z_MAX)
else:
    raise ValueError(f"Unknown DATA_SOURCE='{DATA_SOURCE}'")

print(f"Metadata: {len(df_meta):,} rows × {len(df_meta.columns)} columns")
print(df_meta.head(3).to_string())


## Cell 3 — SQL Builder (for astroquery mode only)

In [ ]:
# This cell is only relevant when DATA_SOURCE = "astroquery".
# In local_fits mode, the SQL was already run by data_download.ipynb.
print("In local_fits mode this cell is informational only.")
print("The SQL used by data_download.ipynb is printed below for reference.")

example_sql = """SELECT TOP {n}
    s.specobjid, s.plate, s.mjd, s.fiberid,
    s.z AS redshift, s.zwarning, s.subclass AS specsubclass, p.petroMag_r
FROM SpecObj AS s JOIN PhotoObj AS p ON s.bestobjid = p.objid
WHERE s.class = 'GALAXY'
  AND s.z BETWEEN 0.01 AND 0.5
  AND s.zwarning = 0
  AND p.petroMag_r BETWEEN 14.0 AND 17.8
  AND p.petroMag_r != -9999
ORDER BY s.specobjid"""

print(example_sql)


## Cell 4 — Load Spectra

### Two modes

**`local_fits` (recommended):**
Reads pre-downloaded `sdss_raw/fits/{plate:04d}/spec-{plate:04d}-{mjd:05d}-{fiber:04d}.fits`
files directly with `astropy.io.fits`. This is ~10× faster than astroquery (no HTTP
overhead), fully resumable, and works offline.

**`astroquery` (fallback for small N):**
Downloads spectra on-the-fly. Useful if you just want to test with a few hundred galaxies
without running data_download.ipynb first.

### FITS file structure

Each SDSS spec-lite FITS file has three extensions:

| Extension | Type | Contents |
|---|---|---|
| 0 | Primary HDU | Observation metadata in header |
| 1 | BinTableHDU | `loglam`, `flux`, `ivar`, `and_mask`, `sky`, … |
| 2 | BinTableHDU | Per-frame (sub-exposure) metadata |

We only need extension 1. The wavelength axis is `loglam` = log₁₀(λ/Å),
log-spaced at Δlog₁₀(λ) = 10⁻⁴ dex/pixel (gives ~69 km/s/pixel velocity resolution).


In [ ]:
def load_one_fits_local(row, fits_dir=FITS_DIR):
    """
    Read a single SDSS spec-lite FITS file from disk.

    Parameters
    ----------
    row : pd.Series
        One row of df_meta with plate, mjd, fiberid, redshift, etc.
    fits_dir : Path
        Root directory of the downloaded FITS tree.

    Returns
    -------
    dict or None
        None if the file is missing or corrupt.
    """
    plate = int(row["plate"])
    mjd   = int(row["mjd"])
    fiber = int(row["fiberid"])

    fpath = fits_dir / f"{plate:04d}" / f"spec-{plate:04d}-{mjd:05d}-{fiber:04d}.fits"

    if not fpath.exists():
        return None   # not downloaded yet

    try:
        with fits.open(fpath, memmap=False) as hdul:
            # Extension 1 is the SpecObj binary table
            data = hdul[1].data
            return {
                "specobjid"   : int(row["specobjid"]),
                "redshift"    : float(row["redshift"]),
                "specsubclass": str(row.get("specsubclass", "")).strip(),
                "petroMag_r"  : float(row["petromag_r"]),
                # Wavelength axis: 10^loglam gives observed-frame Å
                "loglam"      : data["loglam"].astype(np.float64),
                "wavelength"  : (10.0 ** data["loglam"]).astype(np.float64),
                "flux"        : data["flux"].astype(np.float64),
                "ivar"        : data["ivar"].astype(np.float64),
                "andmask"     : data["and_mask"].astype(np.int32),
            }
    except Exception as e:
        # Corrupt file — log and skip
        log.debug(f"Failed to read {fpath}: {type(e).__name__}: {e}")
        return None


def load_spectra_local(df, fits_dir=FITS_DIR):
    """
    Load all available local FITS files for a metadata dataframe.

    Parameters
    ----------
    df : pd.DataFrame
        Rows from df_meta (plate, mjd, fiberid, redshift, etc.)
    fits_dir : Path
        Root of the FITS tree from data_download.ipynb.

    Returns
    -------
    list of dict
        One dict per successfully read spectrum.
    """
    spectra_raw = []
    n_missing = n_corrupt = 0

    for i, row in df.iterrows():
        result = load_one_fits_local(row, fits_dir)
        if result is None:
            p = fits_dir / f"{int(row['plate']):04d}" / f"spec-{int(row['plate']):04d}-{int(row['mjd']):05d}-{int(row['fiberid']):04d}.fits"
            if not p.exists():
                n_missing += 1
            else:
                n_corrupt += 1
        else:
            spectra_raw.append(result)

        if (i + 1) % 2000 == 0:
            log.info(f"  Read {len(spectra_raw):,} / {i+1:,} "
                     f"(missing={n_missing}, corrupt={n_corrupt})")

    log.info(f"Local FITS load complete: {len(spectra_raw):,} OK, "
             f"{n_missing:,} missing, {n_corrupt:,} corrupt")

    if n_missing > 0:
        pct = 100 * n_missing / len(df)
        log.warning(f"{pct:.1f}% of files missing — run data_download.ipynb to fetch them.")

    return spectra_raw


def load_spectra_astroquery(df):
    """Fallback: download spectra live via astroquery."""
    from astroquery.sdss import SDSS
    spectra_raw = []
    n_failed = 0

    for i, row in df.iterrows():
        try:
            fits_list = SDSS.get_spectra(
                plate=int(row["plate"]), mjd=int(row["mjd"]), fiberID=int(row["fiberid"])
            )
            if not fits_list:
                n_failed += 1; continue
            data = fits_list[0][1].data
            spectra_raw.append({
                "specobjid"   : int(row["specobjid"]),
                "redshift"    : float(row["redshift"]),
                "specsubclass": str(row.get("specsubclass","")).strip(),
                "petroMag_r"  : float(row["petromag_r"]),
                "loglam"      : data["loglam"].astype(np.float64),
                "wavelength"  : (10.0 ** data["loglam"]).astype(np.float64),
                "flux"        : data["flux"].astype(np.float64),
                "ivar"        : data["ivar"].astype(np.float64),
                "andmask"     : data["and_mask"].astype(np.int32),
            })
        except Exception as e:
            n_failed += 1
            if n_failed <= 3: log.warning(f"  Row {i}: {e}")
        if (i + 1) % 50 == 0:
            log.info(f"  {i+1}/{len(df)}  ({len(spectra_raw)} OK)")

    log.info(f"astroquery: {len(spectra_raw)} OK, {n_failed} failed")
    return spectra_raw


# ── Run the load ───────────────────────────────────────────────────────
t0 = time.time()
if DATA_SOURCE == "local_fits":
    spectra_raw = load_spectra_local(df_meta, FITS_DIR)
else:
    spectra_raw = load_spectra_astroquery(df_meta)

log.info(f"Loaded {len(spectra_raw):,} spectra in {time.time()-t0:.1f}s")


## Cell 5 — Preprocessing Pipeline

Seven sequential steps produce all output formats from each raw spectrum.
(Identical logic to the previous version of this notebook — only the input
reading has changed.)

| Step | Operation | Why |
|---|---|---|
| 1 | Hardware bad-pixel mask | Remove ivar=0 and andmask≠0 pixels |
| 2 | Interp to spender obs-frame grid | Put spectrum on spender's exact λ axis |
| 3 | De-redshift to rest frame | λ_rest = λ_obs/(1+z) |
| 4 | Additional skyline masking | Only for rest-frame outputs |
| 5 | Compute norm_const | Median flux in 5300–5850 Å (MLP feature #7) |
| 6 | Normalise rest-frame spectrum | Divide by norm_const |
| 7 | Outlier clip (rest-frame only) | ±10 nMAD robust clipping |


In [ ]:
def preprocess_spectrum(raw_spec):
    """
    Process one raw SDSS spectrum into all three output formats.

    Returns dict with keys:
      flux_obs         (float32, 3921)  — observed frame for spender
      flux_rest_unnorm (float32, 3600)  — rest frame, NOT normalised
      flux_rest_norm   (float32, 3600)  — rest frame, normalised
      norm_const       (float32, scalar)
    Returns None if the spectrum fails quality cuts.
    """
    z        = raw_spec["redshift"]
    wave_obs = raw_spec["wavelength"]
    flux     = raw_spec["flux"].copy()
    ivar     = raw_spec["ivar"]
    andmask  = raw_spec["andmask"]

    # ── Step 1: Hardware bad-pixel mask ───────────────────────────────
    bad_hw = (andmask != 0) | (ivar <= 0) | (~np.isfinite(flux))
    flux[bad_hw] = np.nan
    if np.isfinite(flux).sum() < 100:
        return None

    # ── Step 2: Interpolate to spender's observed-frame grid ──────────
    # We use only pixels with valid flux; fill gaps with 0 (not NaN).
    valid = np.isfinite(flux)
    try:
        f_obs = interp1d(wave_obs[valid], flux[valid], kind="linear",
                         bounds_error=False, fill_value=0.0)
        flux_obs = f_obs(wave_obs_grid).astype(np.float32)
    except Exception:
        return None

    # ── Step 3: De-redshift ────────────────────────────────────────────
    wave_rest_native = wave_obs / (1.0 + z)

    # ── Step 4: Additional skyline masking for rest-frame outputs ──────
    bad_sky = bad_hw.copy()
    for sky_lam in SKYLINES_OBS:
        bad_sky |= (np.abs(wave_rest_native - sky_lam / (1.0 + z)) < SKYLINE_WIDTH_AA)
    flux_rest = flux.copy(); flux_rest[bad_sky] = np.nan
    valid_rest = np.isfinite(flux_rest)
    if valid_rest.sum() < 100:
        return None

    try:
        f_rest = interp1d(wave_rest_native[valid_rest], flux_rest[valid_rest],
                          kind="linear", bounds_error=False, fill_value=0.0)
        flux_rest_unnorm = f_rest(wave_rest_grid)
    except Exception:
        return None

    # ── Step 5: Compute norm_const ─────────────────────────────────────
    nm = (wave_rest_grid >= NORM_WAVE_MIN) & (wave_rest_grid <= NORM_WAVE_MAX)
    nv = flux_rest_unnorm[nm]; nv = nv[np.isfinite(nv) & (nv > 0)]
    if len(nv) < 5:
        return None
    norm_const = float(np.median(nv))
    if not np.isfinite(norm_const) or norm_const <= 0:
        return None

    # ── Step 6: Normalise ──────────────────────────────────────────────
    flux_rest_norm = flux_rest_unnorm / norm_const

    # ── Step 7: Outlier clip (rest-frame normalised only) ──────────────
    fm = np.isfinite(flux_rest_norm) & (flux_rest_norm != 0)
    if fm.sum() > 10:
        med  = np.median(flux_rest_norm[fm])
        nmad = 1.4826 * np.median(np.abs(flux_rest_norm[fm] - med))
        flux_rest_norm = np.clip(flux_rest_norm, med - 10*nmad, med + 10*nmad)

    return {
        "flux_obs"         : flux_obs.astype(np.float32),
        "flux_rest_unnorm" : flux_rest_unnorm.astype(np.float32),
        "flux_rest_norm"   : flux_rest_norm.astype(np.float32),
        "norm_const"       : np.float32(norm_const),
    }


In [ ]:
# ── Apply preprocessing ───────────────────────────────────────────────
obs_rows, unnorm_rows, norm_rows, norm_const_list, meta_clean = [], [], [], [], []
n_failed = 0
t0 = time.time()

for i, raw_spec in enumerate(spectra_raw):
    result = preprocess_spectrum(raw_spec)
    if result is None:
        n_failed += 1
        continue
    obs_rows.append(result["flux_obs"])
    unnorm_rows.append(result["flux_rest_unnorm"])
    norm_rows.append(result["flux_rest_norm"])
    norm_const_list.append(result["norm_const"])
    meta_clean.append({
        "specobjid"   : raw_spec["specobjid"],
        "redshift"    : raw_spec["redshift"],
        "specsubclass": raw_spec["specsubclass"],
        "petroMag_r"  : raw_spec["petroMag_r"],
    })
    if (i + 1) % 5000 == 0:
        log.info(f"  Preprocessed {i+1:,}/{len(spectra_raw):,}  "
                 f"({len(norm_rows):,} OK, {n_failed} failed)")

X_obs         = np.vstack(obs_rows)
X_rest_unnorm = np.vstack(unnorm_rows)
X_rest_norm   = np.vstack(norm_rows)
norm_consts   = np.array(norm_const_list, dtype=np.float32)
N             = len(meta_clean)
redshifts     = np.array([m["redshift"]    for m in meta_clean], dtype=np.float32)
specobjids    = np.array([m["specobjid"]   for m in meta_clean], dtype=np.int64)

log.info(f"Done in {time.time()-t0:.1f}s: {N:,} OK, {n_failed} failed "
         f"({100*N/max(1,len(spectra_raw)):.1f}% pass rate)")
print(f"X_obs shape:          {X_obs.shape}   (observed-frame, for spender)")
print(f"X_rest_norm shape:    {X_rest_norm.shape}   (rest-frame, normalised)")
print(f"X_rest_unnorm shape:  {X_rest_unnorm.shape}   (rest-frame, unnormalised)")
print(f"norm_consts shape:    {norm_consts.shape}")


## Cell 6 — Diagnostic Plots

In [ ]:
lines = {"[OII]":3727,"Hβ":4861,"[OIII]":5007,"MgI b":5175,"NaI D":5893,"Hα":6563}

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# Mean normalised rest-frame spectrum
ax = axes[0,0]
mu  = X_rest_norm.mean(axis=0)
p16 = np.percentile(X_rest_norm, 16, axis=0)
p84 = np.percentile(X_rest_norm, 84, axis=0)
ax.plot(wave_rest_grid, mu, "steelblue", lw=1.2, label="Mean")
ax.fill_between(wave_rest_grid, p16, p84, color="steelblue", alpha=0.2, label="16–84th %ile")
ax.axvspan(NORM_WAVE_MIN, NORM_WAVE_MAX, color="gold", alpha=0.2, label="Norm window")
for name, lam in lines.items():
    if WAVE_REST_MIN < lam < WAVE_REST_MAX:
        ax.axvline(lam, ls=":", lw=0.7, color="gray", alpha=0.6)
ax.set_xlabel("Rest-frame λ (Å)"); ax.set_ylabel("Norm. flux")
ax.set_title("Mean normalised rest-frame spectrum")
ax.legend(fontsize=8); ax.set_xlim(WAVE_REST_MIN, WAVE_REST_MAX)

# Mean observed-frame spectrum
ax = axes[0,1]
mu_obs = X_obs.mean(axis=0)
ax.plot(wave_obs_grid, mu_obs, "darkorange", lw=1.2)
ax.set_xlabel("Observed-frame λ (Å)"); ax.set_ylabel("Flux [10⁻¹⁷ erg/s/cm²/Å]")
ax.set_title("Mean observed-frame spectrum (spender input)")

# Redshift distribution
ax = axes[1,0]
ax.hist(redshifts, bins=50, color="seagreen", edgecolor="white")
ax.set_xlabel("Redshift z"); ax.set_ylabel("Count")
ax.set_title("Redshift distribution")

# norm_const distribution
ax = axes[1,1]
ax.hist(np.log10(norm_consts), bins=50, color="tomato", edgecolor="white")
ax.set_xlabel("log₁₀(norm_const) [10⁻¹⁷ erg/s/cm²/Å]"); ax.set_ylabel("Count")
ax.set_title("Normalisation constants (MLP feature #7)")

plt.suptitle(f"SDSS Preprocessing Diagnostics  (N={N:,})", fontweight="bold")
plt.tight_layout()
plt.savefig(OUT_DIR / "sdss_preprocessing_diagnostics.png", dpi=130, bbox_inches="tight")
plt.show()


## Cell 7 — Save All Outputs

In [ ]:
np.save(OUT_DIR / "spectra_obs_frame.npy",       X_obs)
np.save(OUT_DIR / "spectra_restframe_norm.npy",  X_rest_norm)
np.save(OUT_DIR / "spectra_restframe_unnorm.npy",X_rest_unnorm)
np.save(OUT_DIR / "spectra_matrix.npy",          X_rest_norm)   # backwards-compat alias
np.save(OUT_DIR / "norm_consts.npy",             norm_consts)
np.save(OUT_DIR / "wave_obs_grid.npy",           wave_obs_grid)
np.save(OUT_DIR / "wave_grid.npy",               wave_rest_grid)

dtype = np.dtype([("specobjid","i8"),("redshift","f4"),("specsubclass","U32"),("petroMag_r","f4")])
meta_arr = np.empty(N, dtype=dtype)
for field in ["specobjid","redshift","specsubclass","petroMag_r"]:
    meta_arr[field] = [m[field] for m in meta_clean]
np.save(OUT_DIR / "metadata.npy", meta_arr)

print(f"Saved to {OUT_DIR}/:")
for f in sorted(OUT_DIR.iterdir()):
    if f.suffix in (".npy",".npz",".png"):
        print(f"  {f.name:<45s}  {f.stat().st_size/1024:>8.1f} kB")


## Cell 8 — Reload Verification

In [ ]:
checks = {
    "spectra_obs_frame.npy":       ((N, N_PIXELS_OBS),  np.float32),
    "spectra_restframe_norm.npy":  ((N, N_PIXELS_REST), np.float32),
    "spectra_restframe_unnorm.npy":((N, N_PIXELS_REST), np.float32),
    "norm_consts.npy":             ((N,),               np.float32),
    "wave_obs_grid.npy":           ((N_PIXELS_OBS,),    np.float64),
    "wave_grid.npy":               ((N_PIXELS_REST,),   np.float64),
}
all_ok = True
for fname, (eshape, edtype) in checks.items():
    arr = np.load(OUT_DIR / fname)
    ok  = (arr.shape == eshape) and (arr.dtype == edtype)
    print(f"  {'✓' if ok else '✗'} {fname:<45s}  {str(arr.shape):<15s}  {arr.dtype}")
    if not ok: all_ok = False

nc = np.load(OUT_DIR / "norm_consts.npy")
n_bad = (~np.isfinite(nc) | (nc <= 0)).sum()
print(f"\n  {'✓' if n_bad==0 else '✗'} norm_consts: {n_bad} non-positive/non-finite")
print(f"\n{'✓ All checks passed!' if all_ok else '✗ Some checks failed — see above.'}")
